# 02 — First Circuit with IBM Bob

This notebook is one pass through the workshop's core loop:

**See it → Ask Bob → Run it → Read it**

You already *saw* the Bell state in notebook 01. Here you'll **ask IBM Bob** to
write and explain it, **run** the circuit locally on the Aer simulator, and
**read** the result back.

> **Running this notebook** — from the repo root:
> ```bash
> uv run --with jupyterlab --with qiskit==1.2.0 --with qiskit-aer --with matplotlib \
>   --with pylatexenc --with httpx \
>   jupyter lab notebooks/02-first-circuit-with-bob.ipynb
> ```
> For the live "Ask Bob" step, also start the MCP server in a separate terminal:
> ```bash
> cd mcp-server && uv run server.py --http           # IBM Bob's tool, direct HTTP access
> ```
> That step degrades gracefully to a message if the MCP server isn't running. Circuit
> execution below always runs locally — no server needed for it.

## Step 0 — Install dependencies

If you launched with the `uv run …` command above, these are already available and the cell
below is a quick no-op.

In [ ]:
%pip install -q qiskit==1.2.0 qiskit-aer matplotlib pylatexenc httpx
print("✅ Qiskit + Aer + matplotlib + pylatexenc + httpx ready")

## Step 0b — Load the workshop helpers (`notebooks/lib/`)

In [ ]:
import sys
from pathlib import Path

_notebooks_dir = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd() / "notebooks"
if str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

from lib.visuals import draw_circuit, plot_counts, run_and_plot
from lib.backend import run_via_backend, explain_with_bob

## Step 1 — Ask Bob to write the circuit

In IBM Bob's chat, try asking:

> "Write a Qiskit Bell state circuit with 2 qubits and explain what it does."

Bob calls the MCP tool `explain_circuit` — the same Qiskit Code Assistant tool wired into its
MCP config. The cell below calls that exact tool directly over HTTP, so you can see precisely
what Bob sees (it skips gracefully if the MCP server isn't running in `--http` mode).

In [ ]:
bell_code = """from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
"""

explanation = explain_with_bob(bell_code)
if explanation:
    print(explanation)
else:
    print(
        "MCP server not reachable at http://localhost:8765 \u2014 start it with "
        "`cd mcp-server && uv run server.py --http`, or ask Bob directly in "
        "your assistant chat. Continuing with the circuit below either way."
    )

## Step 2 — The circuit Bob wrote

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h(0)           # Hadamard on qubit 0
qc.cx(0, 1)       # CNOT: control=0, target=1
qc.measure([0, 1], [0, 1])

draw_circuit(qc)

## Step 3 — Run it

Same Bell state as notebook 01, but this time it runs through `run_via_backend` on the Aer
simulator — the exact path IBM Bob's "run it" beat uses in the live demo.

In [ ]:
counts = run_via_backend(bell_code)

if counts is None:
    print("Circuit execution failed — falling back to a direct local run.")
    counts = run_and_plot(qc, title="Bell state (local fallback)")
else:
    print("Ran locally on the Aer simulator.")
    plot_counts(counts, title="Bell state")

## Step 4 — Read it

Only `00` and `11` should appear — never `01` or `10`. That's entanglement: measuring one
qubit instantly determines the other.

In [ ]:
print("Counts:", counts)
correlated = set(counts) <= {"00", "11"}
print("Only 00/11 observed:", correlated)

## What's next?

Continue to **notebook 03** to run this same circuit on real IBM Quantum hardware, or visit the
`/demo` route for the full presenter slideshow with animated visualizations.

---
*Quantum Experience Lab — IBM Bob + Qiskit workshop*